In [7]:
import pandas as pd
import numpy as np

# Vos profils de référence strictement calés sur Do = 0
PRESSING_PROFILES = {
    '7-35': {'name': 'Diatonique', 'profile': np.array([1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1])},
    '7-34': {'name': 'Acoustique', 'profile': np.array([1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0])},
    '6-35': {'name': 'Par tons',   'profile': np.array([1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0])},
    '8-28': {'name': 'Octatonique', 'profile': np.array([1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0])},
    '6-20': {'name': 'Hexatonique', 'profile': np.array([1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0])}
}

def identify_set_of_segment(notes_df, start_mc, end_mc):
    """Analyse l'ensemble des notes contenues entre deux mesures choisies par vous."""
    segment = notes_df[(notes_df['mc'] >= start_mc) & (notes_df['mc'] < end_mc)]
    if segment.empty:
        return "Vide", -1
    
    # Extraction brute de toutes les pitch classes de la section
    chroma = np.zeros(12)
    for midi in segment['midi'].dropna():
        chroma[int(midi) % 12] += 1
        
    if chroma.sum() == 0:
        return "Vide", -1
    chroma = chroma / chroma.sum()

    best_set = "Inconnu"
    best_root = -1
    max_corr = -1

    for set_id, info in PRESSING_PROFILES.items():
        for t in range(12):
            transposed = np.roll(info['profile'], t)
            corr = np.corrcoef(chroma, transposed)[0, 1]
            if corr > max_corr:
                max_corr = corr
                best_set = info['name']
                best_root = t
                
    return best_set, best_root

# --- VOTRE TRAVAIL D'OREILLE (L'unique vérité syntaxique) ---
# Vous listez uniquement les mesures où VOUS entendez la bascule structurelle.
# Exemple pour le Clair de lune (mesure de départ de chaque grand espace) :
# Supposons que vous identifiez des ruptures aux mesures 1, 25, 37, 43, 51, et 73 (fin du morceau)
clair_de_lune_breakpoints = [27, 28, 29, 30, 31, 33, 35, 36, 41]

# Chargement du fichier de notes brutes
df_notes = pd.read_csv("datasets/l075-03_suite_clair.notes.tsv", sep='\t')
df_notes['mc'] = pd.to_numeric(df_notes['mc'], errors='coerce').ffill().astype(int)

# Génération automatique du dataset de transition macro
macro_sequence = []
for i in range(len(clair_de_lune_breakpoints) - 1):
    start = clair_de_lune_breakpoints[i]
    end = clair_de_lune_breakpoints[i+1]
    
    pressing_set, root = identify_set_of_segment(df_notes, start, end)
    
    macro_sequence.append({
        'segment_id': i,
        'start_measure': start,
        'end_measure': end,
        'set': pressing_set,
        'root': root  # 1 = Db, 0 = C, etc. Calé proprement grâce au profil sur 0
    })

df_piece_analysis = pd.DataFrame(macro_sequence)
print(df_piece_analysis)

   segment_id  start_measure  end_measure          set  root
0           0             27           28  Hexatonique     0
1           1             28           29  Hexatonique     0
2           2             29           30  Hexatonique     0
3           3             30           31   Diatonique     8
4           4             31           33   Diatonique     1
5           5             33           35   Acoustique     8
6           6             35           36  Hexatonique     0
7           7             36           41   Diatonique     4
